## Bland-Altman analysis generalized to multiple repetitions to characterize SC reliability

### Simulate SC matrices with a set bias

We will use simulated SC matrices with an artifical set bias in the between-session variability to develop a Bland-Altman analysis capable of detecting such biases. Once the methodology is refined, we will apply it on the real HCPh data.

First, we generate a reference SC matrix on which we will later add noise to simulate between-session variability.

In [4]:
import numpy as np

atlas_dim = 64  # Number of brain regions
scale = 0.01  # Mean track density
shape = 0.5  # Variability in density

# SC matrices typically follow a heavy-tailed distribution, with a few strong connections and many weak ones. 
# We thus use a log-normal distribution to simulate SC value.
SC_matrix = np.random.lognormal(mean=np.log(scale), sigma=shape, size=(atlas_dim, atlas_dim))

# Make it symmetric
SC_matrix = (SC_matrix + SC_matrix.T) / 2

# Set diagonal to 0 (no self-connections)
np.fill_diagonal(SC_matrix, 0)

print(SC_matrix.shape)
print(SC_matrix)

(64, 64)
[[0.         0.01060361 0.00918417 ... 0.01205325 0.02164462 0.0071622 ]
 [0.01060361 0.         0.01149394 ... 0.01176145 0.01685142 0.00797059]
 [0.00918417 0.01149394 0.         ... 0.00616541 0.02270113 0.00753738]
 ...
 [0.01205325 0.01176145 0.00616541 ... 0.         0.0099065  0.01933721]
 [0.02164462 0.01685142 0.02270113 ... 0.0099065  0.         0.00899279]
 [0.0071622  0.00797059 0.00753738 ... 0.01933721 0.00899279 0.        ]]


We will duplicate the reference SC matrix 36 times to simulate the 36 acquisition sessions adding some noise in the duplicates to simulate between-session variability.

In the first simulation, we will make the connections with a lower density more variable across sessions.

In [7]:
# Number of sessions
num_sessions = 36

# Create a 3D array to store the SC matrices for all sessions
SC_matrices = np.zeros((atlas_dim, atlas_dim, num_sessions))

# Add noise to each duplicate
noise_level = 0.0005  # Standard deviation of the noise
for i in range(num_sessions):
    noise = np.random.normal(loc=0, scale=noise_level, size=SC_matrix.shape)
    SC_matrices[:, :, i] = SC_matrix + noise

print(SC_matrices.shape)

# Identify the 20th percentile threshold for the connection values in the reference SC matrix
percentile_20_threshold = np.percentile(SC_matrix[SC_matrix > 0], 20)

# Add noise to each duplicate
for i in range(num_sessions):
    noise = np.random.normal(loc=0, scale=noise_level, size=SC_matrix.shape)
    higher_noise = np.random.normal(loc=0, scale=0.002, size=SC_matrix.shape)
    
    # Apply stronger noise to connections below the 20th percentile
    noise[SC_matrix <= percentile_20_threshold] += higher_noise[SC_matrix <= percentile_20_threshold]
    
    SC_matrices[:, :, i] = SC_matrix + noise

print(SC_matrices.shape)

(64, 64, 36)
